# HW2 on H100 (Colab)

Runs all sections end-to-end. Each cell writes its outputs to `logs/<category>/<ts>_<name>/`.

**Before you start:** Runtime → Change runtime type → **H100 GPU** → Save.

## 1. Setup

Clone, install everything, then **restart the runtime** (vLLM downgrades torch — Python kernel must re-import). Colab will prompt automatically; if not, do it manually after this cell.

In [ ]:
!git clone https://github.com/trevorbchen/148bhw2.git || (cd 148bhw2 && git pull)
%cd /content/148bhw2

# vLLM pins torch==2.5.1 — install it first so it dictates the torch version,
# then install basics with --no-deps so it doesn't try to pull torch 2.6.
# transformers MUST be pinned to ~4.48: vLLM 0.7.2 reads
# tokenizer.all_special_tokens_extended which was removed in newer transformers.
!pip install -q vllm==0.7.2 transformers==4.48.3 datasets pandas pyarrow \
    latex2sympy2-extended "math-verify[antlr4-13-2]" pylatexenc==2.10 sympy \
    humanfriendly wandb einops einx jaxtyping tiktoken regex psutil submitit
!pip install -q -e ./basics --no-deps

import os
os.kill(os.getpid(), 9)  # force restart so the new torch is picked up

## 2. After restart — sanity check

Run this first after the kernel restarts.

In [ ]:
%cd /content/148bhw2
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
import basics, vllm, transformers
print("basics, vllm, transformers all import OK")

## 3. Systems benchmarks (Section 2.3-2.6)

Sweeps across model sizes + modes. Each run goes to its own `logs/systems/<ts>_*/` folder.

In [ ]:
# 2.3: forward / forward-backward / train-step across sizes
for size in ["small", "medium", "large"]:
    for mode in ["forward", "forward-backward", "train-step"]:
        !python -m systems.benchmark --model-size {size} --mode {mode}

In [ ]:
# 2.3 (3): warmup sweep. Run the medium model under different warmup counts so the
# write-up can compare 0 / 1 / 2 / 5 / 10 warmup steps.
for w in [0, 1, 2, 5, 10]:
    !python -m systems.benchmark --model-size medium --mode forward-backward \
        --warmup-steps {w} --measure-steps 10

In [ ]:
# 2.4: NVIDIA Nsight Systems profiling.
# Writes .nsys-rep files into logs/systems/nsys/. Download them and open in the
# Nsight Systems desktop app (free: https://developer.nvidia.com/nsight-systems).
# The PDF deliverables (top kernel, softmax vs matmul, kernel mix in train step,
# etc.) are answered by inspecting the GUI and filtering by NVTX ranges.

import os, shutil, glob

def _find_nsys():
    p = shutil.which('nsys')
    if p:
        return p
    cands = sorted(
        glob.glob('/usr/local/cuda*/bin/nsys')
        + glob.glob('/opt/nvidia/**/nsys', recursive=True)
        + glob.glob('/usr/local/bin/nsys')
    )
    return cands[-1] if cands else None

nsys = _find_nsys()
if nsys is None:
    print('nsys not found — installing newest nsight-systems-* package via apt...')
    !apt-get update -q
    # 'nsight-systems' is virtual on NVIDIA's repo; install a concrete versioned package.
    !apt-get install -y $(apt-cache search '^nsight-systems-' | sort -V | tail -1 | awk '{print $1}') \
        || apt-get install -y cuda-nsight-systems-12-4
    nsys = _find_nsys()

if nsys is None:
    raise RuntimeError('Could not locate nsys after install attempt')

os.environ['PATH'] = os.path.dirname(nsys) + ':' + os.environ['PATH']
print('Using nsys:', nsys)
!nsys --version

!mkdir -p logs/systems/nsys
# Trim sizes / contexts / modes below to a subset to save Colab time.
# `-t cuda,nvtx,osrt` works on all nsys versions (avoids the newer --pytorch flag).
for size in ['small', 'medium', 'large']:
    for ctx in [32, 64, 128, 256]:
        for mode in ['forward', 'forward-backward', 'train-step']:
            tag = f'{size}_ctx{ctx}_{mode}'
            print(f'\n=== nsys: {tag} ===')
            !nsys profile -t cuda,nvtx,osrt --force-overwrite=true \
                -o logs/systems/nsys/{tag} \
                python -m systems.benchmark \
                --model-size {size} --context-length {ctx} --mode {mode} \
                --warmup-steps 1 --measure-steps 3 --nvtx 2>&1 | tail -10

In [ ]:
# 2.6: memory profiler on the LARGE model (PDF p8) across context lengths.
# (1) two timelines: forward-only and full train-step  (open the .pickle in https://pytorch.org/memory_viz)
# (2) peak memory per context length, fwd vs train-step (table)
# (3) bf16 mixed precision memory comparison
# Each run dir gets memory.pickle + results.json. results.json's max/mean tells you peak.
for ctx in [128, 256, 512]:
    !python -m systems.benchmark --model-size large --mode forward \
        --context-length {ctx} --use-memory-profiler --measure-steps 3
    !python -m systems.benchmark --model-size large --mode train-step \
        --context-length {ctx} --use-memory-profiler --measure-steps 3
    !python -m systems.benchmark --model-size large --mode train-step \
        --context-length {ctx} --use-memory-profiler --use-bf16 --measure-steps 3

## 4. Attention benchmark (Section 2.7-2.8)

In [ ]:
# §2.7 + §2.8 (1): attention sweep, eager and torch.compile.
# Grid: head_dim ∈ {16,32,64,128} × seq_len ∈ {64,128,256,512,1024}, batch=8.
# OOM rows are tagged "error": "OOM" in results.json (PDF asks: at what size do you OOM?).
!python -m systems.attention_benchmark
!python -m systems.attention_benchmark --compile-attention

In [ ]:
# 2.8 (2): compile the WHOLE transformer (not just attention) and compare to vanilla.
# Pairs with the §2.3 baseline runs above so the table can compare per (size, mode).
for size in ["small", "medium", "large"]:
    for mode in ["forward", "forward-backward", "train-step"]:
        !python -m systems.benchmark --model-size {size} --mode {mode} --compile-model

## 5. Alignment baselines (Section 3.1-3.2)

First run downloads Qwen2.5-Math-1.5B (~3GB).

Defaults (per PDF p13):
- `--split train` — Problem 1 says "make sure to load the train split"
- Reward fn auto-picked per mode: `answer_tag` for direct, `r1_zero` for cot/self_consistency
- Override either with `--split test` or `--reward-fn r1_zero|answer_tag`

In [ ]:
# PDF p13: defaults to --split train + r1_zero_reward_fn for cot/self_consistency
# (direct stays on answer_tag since the direct prompt has no <think> block).
# Override with --split test or --reward-fn r1_zero/answer_tag.
# Drop --limit for the full sweep (train: ~7473 examples).
!python -m alignment.eval --mode direct --limit 256
!python -m alignment.eval --mode cot --limit 256
!python -m alignment.eval --mode self_consistency --k 5 --limit 256

## 6. GRPO training (Section 3.5)

PDF deliverables require **two 50-step runs** to compare normalize-by-std True vs False (§3.5 Problem grpo group standard deviation). Each run is ~30-60 min on H100, longer on lower-tier GPUs.

Validation rewards are eval'd every 5 steps — used by the plot cell below.

In [ ]:
# §3.5 (grpo train loop): 50 iterations with normalize_by_std=True (default).
# Eval val reward every 5 steps; rollouts logged to generations.jsonl.
!python -m alignment.grpo \
    --n-grpo-steps 50 \
    --rollout-batch-size 32 \
    --group-size 8 \
    --train-batch-size 32 \
    --gradient-accumulation-steps 16 \
    --eval-every 5 \
    --n-eval-examples 256 \
    --run-name grpo_std_true

In [ ]:
# §3.5 (grpo group standard deviation): same config but normalize_by_std=False.
!python -m alignment.grpo \
    --n-grpo-steps 50 \
    --rollout-batch-size 32 \
    --group-size 8 \
    --train-batch-size 32 \
    --gradient-accumulation-steps 16 \
    --eval-every 5 \
    --n-eval-examples 256 \
    --no-normalize-by-std \
    --run-name grpo_std_false

In [ ]:
# §3.5 plot: validation rewards over steps for both runs.
# Saves to logs/alignment/grpo_val_curves.png — include in writeup.
import json, glob
from pathlib import Path
import matplotlib.pyplot as plt

def _latest_run(name: str) -> Path | None:
    matches = sorted(glob.glob(f"logs/alignment/*_{name}"))
    return Path(matches[-1]) if matches else None

def _curve(run_dir: Path) -> tuple[list[int], list[float], list[float]]:
    steps, ans, fmt = [], [], []
    for line in (run_dir / "train_log.jsonl").read_text().splitlines():
        row = json.loads(line)
        if "val/answer_reward" in row:
            steps.append(row["step"])
            ans.append(row["val/answer_reward"])
            fmt.append(row["val/format_reward"])
    return steps, ans, fmt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, run_name in [("normalize_by_std=True", "grpo_std_true"), ("normalize_by_std=False", "grpo_std_false")]:
    run = _latest_run(run_name)
    if run is None:
        print(f"skip {label}: no run found")
        continue
    s, a, f = _curve(run)
    axes[0].plot(s, a, marker="o", label=label)
    axes[1].plot(s, f, marker="o", label=label)

axes[0].set(xlabel="step", ylabel="val/answer_reward", title="GRPO: validation answer reward")
axes[1].set(xlabel="step", ylabel="val/format_reward", title="GRPO: validation format reward")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()

out = Path("logs/alignment/grpo_val_curves.png")
plt.tight_layout(); plt.savefig(out, dpi=120)
plt.show()
print(f"Saved {out}")

## 7. Bundle logs and download

Colab VMs are ephemeral — pull the logs back to your laptop before the session dies. Includes the `grpo_val_curves.png` plot.

In [ ]:
!ls -R logs/ | head -60
# Exclude saved model weights (final/) — they're huge and not needed for the writeup
!cd logs && zip -r /content/logs.zip . -x '*final/*'
from google.colab import files
files.download('/content/logs.zip')